# 03_2d_inversion (Voila)

Stage inversion inputs, run one multi-scale ladder, watch live progress, and compare against the true model.

Generate inputs writes a uniform `sg0` per frequency under `workspace/2D/inversion/input/`. Run inversion allocates one `Run{N}/` for the whole ladder; each frequency is a subdirectory that is the engine cwd. Lowest frequency first; later scales start from the previous inverted model, resampled onto their own grid.

Run all cells, then launch with Voila (`--strip_sources=True`).

In [ ]:
from pathlib import Path
import json
import os
import re
import signal
import sys
import threading
import time
import traceback
import subprocess

# Project root: start scripts run from repo root, so cwd is the workshop directory
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.modules.workshop_config import load_config, patch_runinv_template
CONFIG = load_config()
WORKSPACE = CONFIG.workspace

try:
    import ipywidgets as ipw
    import numpy as np
    import plotly.graph_objects as go
except Exception as exc:
    raise RuntimeError(
        'Missing GUI dependencies. Install with: pip install voila ipywidgets plotly numpy ipykernel matplotlib scipy segyio'
    ) from exc

from scripts.modules.inversion import (
    apply_inversion_controls, find_observed_records, prepare_inversion_inputs,
    stage_run_directory,
)
from scripts.modules.multiscale_2d import (
    allocate_ladder_run, empty_ladder_manifest, find_output_model,
    handoff_starting_model, iter_stages, ladder_run_label, list_run_dirs,
    stage_cwd, upsert_stage, write_ladder_manifest,
)
from scripts.modules.fd_visualization import load_rss_traces

# `display` is a KERNEL-INJECTED builtin in a live notebook, so cell 2's
# `display(layout)` worked here for a long time without ever being imported -
# and was a NameError in every headless driver (validate_notebooks, chainsweep
# and handlersweep each had to inject it to run this notebook at all). Import it
# explicitly, like notebooks 02/04/05/06 do, so nothing depends on the kernel.
from IPython.display import display, clear_output

# --- acquisition matrix -----------------------------------------------------
# Step 01 emits one dataset per (frequency, source component) pair. This step
# inverts them one FREQUENCY at a time, JOINTLY over the sources:
# `mpiEminvTE2d` takes a comma-separated `source_type`, so its work list spans
# (shot x source type) and every source stacks into the same gradient. A 4-
# component acquisition (Kx/Kz sources x Hx/Hz receivers) is therefore ONE
# inversion per frequency whose every model update sees all four components,
# not two runs where the second overwrites the first's answer.
#
# The sequence over frequencies is a ladder, lowest first. Frequency 1 starts
# from the uniform `sg0` that `prepare_inversion_inputs` writes; every later
# frequency starts from the previous inverted model (`Results/sg_up.rss-<iter>`),
# resampled onto this frequency's own grid by `handoff_starting_model`. That is
# why the frequencies run sequentially: each needs the model the previous one
# wrote. One click of Run inversion allocates one `Run{N}/` and puts each
# frequency in a subdirectory; `mpiEminvTE2d` cwd is that subdirectory.
#
# `DS` still points at ONE dataset, because the per-dataset artifacts a joint
# run reads - sg.rss, ep.rss, setup_metadata.json - are shared by a frequency's
# sources (`build_forward_matrix` varies only `source_field`), and
# `prepare_inversion_inputs` asserts they agree. Reading `CONFIG.fwd_2d_dir`
# directly sees only the single-dataset layout, and on a matrix workspace there
# is no sg.rss there at all.
#
# ONE mechanism for dataset paths, shared by every notebook:
# `headless.dataset_paths` resolves every per-dataset artifact from a run
# directory, and `DS` is the only global that has to be rebound. Loose
# constants like `FWD_2D_DIR` and `sg_true`, rebound through a hand-maintained
# `global` list, are the pattern that leaves a path pinned to the forward ROOT.
# See `headless.DatasetPaths`.
FWD_ROOT = CONFIG.fwd_2d_dir


def available_groups():
    """The units this step acts on: one per frequency, all its sources together.

    `iter_datasets` returns the (frequency, source) datasets, which is the right
    unit for forward modelling and calibration. A joint inversion's unit is the
    frequency, so the grouping lives in `headless` and is shared with
    `multiscale_2d.build_ladder` rather than being re-derived here.
    """
    from scripts.modules.headless import group_datasets_by_frequency
    try:
        return group_datasets_by_frequency(FWD_ROOT)
    except Exception:
        return []


def _select_dataset(name=None):
    """Bind `DS` to one dataset of the matrix; returns its manifest entry."""
    global DS
    from scripts.modules.headless import select_dataset
    chosen, DS = select_dataset(FWD_ROOT, name)
    return chosen


# Bind DS at import, so nothing below can read an unbound name and no
# root-bound placeholder ever exists for a later edit to leave behind.
_select_dataset(None)

SCRIPTS_DIR = ROOT / 'scripts'
TEMPLATES_DIR = SCRIPTS_DIR / 'templates'
INV_TEMPLATE = TEMPLATES_DIR / 'inv.cfg'
RUNINV_TEMPLATE = TEMPLATES_DIR / 'runinv.sh'
MAX_CPUS = max(2, os.cpu_count() or 2)

INV_2D_INPUT_DIR = CONFIG.inv_2d_input_dir
INV_2D_RUNS_DIR = CONFIG.inv_2d_runs_dir
VOILA_PID_FILE = ROOT / '.voila_2d_inversion_server.pid'
PCT_RE = re.compile(r'(\d+(?:\.\d+)?)%')

state = {
    'process': None,
    'refresh_loop': False,
    'monitor_thread': None,
    'last_messages': [],
    'active_run_dir': None,
    'ladder_run_dir': None,
    'active_run_log': None,
    'last_sg_ls_mtime': None,
    'model_plot_key': None,
    'updating_run_selector': False,
    'refresh_in_progress': False,
    'last_plot_run_dir': None,
    'last_progress_event': None,
    'initial_load': False,
}

def push_message(msg):
    state['last_messages'].append(msg)
    if len(state['last_messages']) > 12:
        state['last_messages'] = state['last_messages'][-12:]
    status_out.value = '\n'.join(state['last_messages'])


def bind_button_with_feedback(button, handler, action_label, success_message=None):
    def _wrapped(_):
        before = len(state.get('last_messages', []))
        push_message(f'{action_label}...')
        try:
            handler(_)
            after = len(state.get('last_messages', []))
            if after <= before + 1:
                push_message(success_message or f'{action_label} completed successfully.')
        except Exception as exc:
            push_message(f'{action_label} failed: {exc}')
            raise

    button.on_click(_wrapped)


def read_tail(path, n_lines=50):
    if path is None:
        return 'No log file yet.'
    p = Path(path)
    if not p.exists():
        return 'No log file yet.'
    lines = p.read_text(errors='replace').splitlines()
    return '\n'.join(lines[-n_lines:]) if lines else 'Log file is empty.'


def read_nonempty_last_line(path):
    p = Path(path)
    if not p.exists():
        return ''
    lines = p.read_text(errors='replace').splitlines()
    for line in reversed(lines):
        if line.strip():
            return line.strip()
    return ''


def find_latest_run_dir(root_dir):
    dirs = list_run_dirs(root_dir)
    if not dirs:
        return None
    return dirs[-1][1]


def update_info_panel():
    active = state.get('active_run_dir')
    active_str = str(active) if active else 'Not started yet'
    info.value = (
        f'<b>Template:</b> {INV_TEMPLATE}<br>'
        f'<b>Inputs folder:</b> {INV_2D_INPUT_DIR}<br>'
        f'<b>Active scale folder:</b> {active_str}<br>'
        f'<b>Ladder:</b> {state.get("ladder_run_dir") or "Not started yet"}<br>'
        f'<b>Run script template:</b> {RUNINV_TEMPLATE}<br>'
        'One Run is one frequency ladder, jointly over the sources of each '
        'tone. The first scale starts from the uniform model; every later '
        'scale starts from the previous inverted model, resampled onto its '
        'own grid. Max iter is a cap on every scale; geps &gt; 0 stops a '
        'scale when GNORM in progress.log falls below it.'
    )


def parse_progress_summary(run_dir):
    if run_dir is None:
        return 'No active run selected.'
    run_dir = Path(run_dir)
    progress_path = run_dir / 'progress.log'
    if not progress_path.exists():
        return 'progress.log not found yet.'

    lines = progress_path.read_text(errors='replace').splitlines()
    max_iter = None
    max_ls = None
    line_rows = []
    last_note = ''

    for line in lines:
        if 'Maximum number of iterations:' in line:
            try:
                max_iter = int(line.split(':', 1)[1].strip())
            except Exception:
                pass
        if 'Maximum number of linesearches:' in line:
            try:
                max_ls = int(line.split(':', 1)[1].strip())
            except Exception:
                pass
        if line.startswith('Linesearch') or line.startswith('Iteration'):
            line_rows.append(line)
        if (
            'Maximum number of iterations is performed' in line
            or 'Maximum number of linesearches is performed' in line
            or 'Too small change in gradient norm' in line
            or 'Functional is minimized' in line
            or 'Maximum number of functional evaluations is performed' in line
        ):
            last_note = line.strip()

    last_iteration = None
    linesearch_in_current_iteration = 0
    for row in line_rows:
        if row.startswith('Iteration'):
            m = re.match(r'^Iteration\s+(\d+)', row)
            if m:
                last_iteration = int(m.group(1))
            linesearch_in_current_iteration = 0
        elif row.startswith('Linesearch'):
            linesearch_in_current_iteration += 1

    total_linesearches = sum(1 for r in line_rows if r.startswith('Linesearch'))
    sg_up_files = sorted(run_dir.glob('Results/sg_up.rss-*'))
    accepted_iters = len(sg_up_files)

    current_iteration_display = last_iteration if last_iteration is not None else 0
    if (last_iteration is None) and total_linesearches > 0:
        current_iteration_display = 1

    if last_note:
        linesearch_in_current_iteration = 0

    last_row = line_rows[-1] if line_rows else 'No iteration/linesearch row yet.'

    return (
        f'Progress file: {progress_path.name}\n'
        f'Max iterations: {max_iter if max_iter is not None else "N/A"}\n'
        f'Max linesearches/iteration: {max_ls if max_ls is not None else "N/A"}\n'
        f'Current reached iteration: {current_iteration_display}\n'
        f'Linesearches attempted in current iteration: {linesearch_in_current_iteration}\n'
        f'Total linesearch evaluations: {total_linesearches}\n'
        f'Accepted iteration models in Results/: {accepted_iters}\n'
        f'Latest progress row:\n{last_row}\n'
        f'{last_note}'
    )


def parse_mpiqueue_summary(run_dir):
    if run_dir is None:
        return ('No active run selected.', 'No active run selected.')
    qpath = Path(run_dir) / 'mpiqueue.log'
    if not qpath.exists():
        return ('mpiqueue.log not found yet.', 'mpiqueue.log not found yet.')

    text = qpath.read_text(errors='replace')
    lines = text.splitlines()
    jobs_total = None
    jobs_remaining = None
    for line in lines:
        m = re.search(r'Jobs:\s*(\d+),\s*#Remaining jobs:\s*(\d+)', line)
        if m:
            jobs_total = int(m.group(1))
            jobs_remaining = int(m.group(2))

    summary = [
        f'Queue file: {qpath.name}',
        f'Jobs total: {jobs_total if jobs_total is not None else "N/A"}',
        f'Jobs remaining: {jobs_remaining if jobs_remaining is not None else "N/A"}',
    ]
    return ('\n'.join(summary), read_tail(qpath, n_lines=30))


def parse_worker_logs_summary(run_dir):
    if run_dir is None:
        return 'No active run selected.'
    logs = sorted(Path(run_dir).glob('log.txt-*'), key=lambda p: p.name)
    if not logs:
        return 'No worker logs found yet.'

    out = []
    for lp in logs:
        last = read_nonempty_last_line(lp)
        pct_match = PCT_RE.search(last)
        pct = pct_match.group(1) + '%' if pct_match else 'N/A'
        status = 'done' if 'completed' in last.lower() or '100%' in last else 'running'
        out.append(f'{lp.name}: {pct} ({status}) :: {last}')
    return '\n'.join(out)


def latest_sg_up_file(run_dir):
    run_dir = Path(run_dir)
    candidates = list(run_dir.glob('Results/sg_up.rss-*')) + list(run_dir.glob('sg_up.rss-*'))
    if not candidates:
        return None

    def _suffix_num(path):
        m = re.search(r'sg_up\.rss-(\d+)$', path.name)
        return int(m.group(1)) if m else -1

    return sorted(candidates, key=lambda p: (_suffix_num(p), p.name))[-1]


def summarize_linesearch_model(run_dir):
    if run_dir is None:
        return 'No active run selected.'
    run_dir = Path(run_dir)
    sg_ls = run_dir / 'sg_ls.rss'
    latest = latest_sg_up_file(run_dir)

    msg = []
    if sg_ls.exists():
        st = sg_ls.stat()
        msg.append(f'Current linesearch model: {sg_ls.name} ({st.st_size} bytes)')
    else:
        msg.append('Current linesearch model: not available yet (sg_ls.rss missing).')

    if latest is not None:
        st = latest.stat()
        msg.append(f'Latest accepted model: {latest.name} ({st.st_size} bytes)')
    else:
        msg.append('Latest accepted model: not available yet (Results/sg_up.rss-* missing).')

    return '\n'.join(msg)


def _read_rss_model(path):
    from third_party.rockseis.io.rsfile import rsfile

    f = rsfile()
    f.read(str(path))

    # Model RSS files are stored as x,y,z. For 2D models y is singleton.
    # Squeeze y and transpose so plotting uses z(rows) x x(columns).
    data = np.asarray(f.data, dtype=float)
    data = np.squeeze(data)
    if data.ndim != 2:
        raise ValueError(f'Expected 2D model RSS after squeeze for {path}, got shape {data.shape}')

    nx = int(data.shape[0])
    nz = int(data.shape[1])
    grid = np.asarray(data.T, dtype=float)

    dx = float(f.geomD[0]) if f.geomD[0] else 1.0
    ox = float(f.geomO[0])

    # z is dimension 2 for x,y,z; for already-2D files fall back to dim 1.
    iz = 2 if (len(f.geomN) > 2 and int(f.geomN[2]) > 0) else 1
    dz = float(f.geomD[iz]) if f.geomD[iz] else 1.0
    oz = float(f.geomO[iz])

    x = ox + dx * np.arange(nx)
    z = oz + dz * np.arange(nz)
    return x, z, grid


def _extract_positions(run_dir):
    run_dir = Path(run_dir)
    # Both staged naming forms: a single-source run's Hx_data.rss and a joint
    # run's Hx_Hx_data.rss. Every record of a run shares the geometry (the
    # engine refuses to start otherwise), so any of them gives the positions.
    obs = find_observed_records(run_dir)
    hx_path = obs.get('HX')
    if hx_path is None:
        return np.array([]), np.array([]), np.array([]), np.array([])
    try:
        meta = load_rss_traces(hx_path)
        src = np.column_stack((np.asarray(meta['src_x'], dtype=float), np.asarray(meta['src_z'], dtype=float)))
        rec = np.column_stack((np.asarray(meta['rx_x'], dtype=float), np.asarray(meta['rx_z'], dtype=float)))
        src_u = np.unique(np.round(src, 6), axis=0)
        rec_u = np.unique(np.round(rec, 6), axis=0)
        return src_u[:, 0], src_u[:, 1], rec_u[:, 0], rec_u[:, 1]
    except Exception:
        return np.array([]), np.array([]), np.array([]), np.array([])


def _align_axes_to_survey(x, z, tx_x, tx_z, rx_x, rx_z):
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)
    pts_x = np.concatenate([tx_x, rx_x]) if (tx_x.size + rx_x.size) > 0 else np.array([])
    pts_z = np.concatenate([tx_z, rx_z]) if (tx_z.size + rx_z.size) > 0 else np.array([])
    if pts_x.size == 0 or pts_z.size == 0:
        return x, z

    x_span = float(np.nanmax(x) - np.nanmin(x)) if x.size else 0.0
    z_span = float(np.nanmax(z) - np.nanmin(z)) if z.size else 0.0
    x_center = float(0.5 * (np.nanmin(x) + np.nanmax(x))) if x.size else 0.0
    z_center = float(0.5 * (np.nanmin(z) + np.nanmax(z))) if z.size else 0.0
    sx_center = float(0.5 * (np.nanmin(pts_x) + np.nanmax(pts_x)))
    sz_center = float(0.5 * (np.nanmin(pts_z) + np.nanmax(pts_z)))

    # If survey and model centers are far apart, shift model axes for plotting.
    if x_span > 0.0 and abs(sx_center - x_center) > 0.5 * x_span:
        x = x + (sx_center - x_center)
    if z_span > 0.0 and abs(sz_center - z_center) > 0.5 * z_span:
        z = z + (sz_center - z_center)
    return x, z


def run_dataset_name(run_dir):
    """The forward group this SCALE inverted, from `<scale>/dataset.txt`."""
    if run_dir is None:
        return None
    f = Path(run_dir) / 'dataset.txt'
    if not f.exists():
        return None
    return f.read_text().strip() or None


def run_true_model(run_dir):
    """The true `sg.rss` of the run's OWN dataset.

    Each frequency is modelled on its own grid, so the true model is a different
    array per dataset - 1.4 m at 2 kHz, 0.8 m at 6 kHz. `DS` is bound to the
    first dataset in the matrix and does not follow the run selector, so reading
    `DS.sg` here plotted the 2 kHz Earth beside a 6 kHz inversion and, because
    the colour limits are taken from it, scaled every panel to the wrong model.
    Resolved by NAME through the same grouping the staging batch used.
    """
    name = run_dataset_name(run_dir)
    if name:
        from scripts.modules.headless import group_datasets_by_frequency
        try:
            for grp in group_datasets_by_frequency(FWD_ROOT):
                if grp['name'] == name:
                    return Path(next(iter(grp['forward_dirs'].values()))) / 'sg.rss'
        except Exception:
            pass
        for entry in _all_datasets():
            if entry.get('name') == name:
                return Path(entry['run_dir']) / 'sg.rss'
    return DS.sg


def _all_datasets():
    from scripts.modules.headless import iter_datasets
    try:
        return iter_datasets(FWD_ROOT)
    except Exception:
        return []


def _current_iteration_model(run_dir):
    latest = latest_sg_up_file(run_dir)
    if latest is not None:
        return latest
    fallback = Path(run_dir) / 'sg0.rss'
    return fallback if fallback.exists() else None


def update_model_plot(run_dir, force=False):
    if run_dir is None:
        return
    run_dir = Path(run_dir)

    # If user switched runs, clear previous run's plot immediately.
    if state.get('last_plot_run_dir') != str(run_dir):
        with model_plot_out:
            clear_output(wait=True)
        state['model_plot_key'] = None

    sg_ls = run_dir / 'sg_ls.rss'
    sg_true = run_true_model(run_dir)
    sg_iter = _current_iteration_model(run_dir)
    sg_up = latest_sg_up_file(run_dir)

    if (not sg_ls.exists()) and (sg_iter is None) and (not sg_true.exists()):
        with model_plot_out:
            clear_output(wait=True)
        state['last_plot_run_dir'] = str(run_dir)
        return

    key = (
        str(sg_ls) if sg_ls.exists() else None,
        sg_ls.stat().st_mtime if sg_ls.exists() else None,
        str(sg_iter) if sg_iter is not None else None,
        sg_iter.stat().st_mtime if sg_iter is not None else None,
        str(sg_up) if sg_up is not None else None,
        sg_up.stat().st_mtime if sg_up is not None else None,
        sg_true.stat().st_mtime if sg_true.exists() else None,
    )
    if (not force) and state.get('model_plot_key') == key:
        return

    tx_x, tx_z, rx_x, rx_z = _extract_positions(run_dir)

    models = []
    if sg_iter is not None:
        models.append(('Current iteration model', sg_iter))
    if sg_ls.exists():
        models.append(('Current linesearch model (sg_ls.rss)', sg_ls))
    if sg_true.exists():
        models.append((f'True conductivity model ({sg_true.parent.name}/sg.rss)', sg_true))
    if sg_up is not None:
        models.append((f'Latest accepted model ({sg_up.name})', sg_up))

    grids = []
    for title, path in models:
        x, z, g = _read_rss_model(path)
        x, z = _align_axes_to_survey(x, z, tx_x, tx_z, rx_x, rx_z)
        grids.append((title, path, x, z, g))

    if not grids:
        return

    # Keep fixed limits from true model when available.
    if sg_true.exists():
        _, _, gtrue = _read_rss_model(sg_true)
        zmin = float(np.nanmin(gtrue))
        zmax = float(np.nanmax(gtrue))
    else:
        gmins = [float(np.nanmin(g)) for _, _, _, _, g in grids]
        gmaxs = [float(np.nanmax(g)) for _, _, _, _, g in grids]
        zmin = min(gmins)
        zmax = max(gmaxs)

    from plotly.subplots import make_subplots

    nplots = len(grids)
    rows = int(np.ceil(nplots / 2))
    fig = make_subplots(rows=rows, cols=2, subplot_titles=[t for t, *_ in grids] + [''] * (rows * 2 - nplots))

    for i, (title, path, x, z, grid) in enumerate(grids):
        row = i // 2 + 1
        col = i % 2 + 1
        fig.add_trace(
            go.Heatmap(
                x=x,
                y=z,
                z=grid,
                colorscale='Viridis',
                zmin=zmin,
                zmax=zmax,
                showscale=(i == 0),
                colorbar=dict(title='S/m'),
            ),
            row=row,
            col=col,
        )
        if tx_x.size:
            fig.add_trace(
                go.Scatter(
                    x=tx_x,
                    y=tx_z,
                    mode='markers',
                    marker=dict(symbol='triangle-up', size=7, color='red'),
                    name='TX',
                    showlegend=(i == 0),
                ),
                row=row,
                col=col,
            )
        if rx_x.size:
            fig.add_trace(
                go.Scatter(
                    x=rx_x,
                    y=rx_z,
                    mode='markers',
                    marker=dict(symbol='circle', size=6, color='cyan'),
                    name='RX',
                    showlegend=(i == 0),
                ),
                row=row,
                col=col,
            )

        fig.update_xaxes(title_text='x (m)', row=row, col=col)
        fig.update_yaxes(title_text='z (m)', autorange='reversed', row=row, col=col)

    fig.update_layout(
        title='Inversion conductivity models (iteration / linesearch / true / latest update)',
        height=max(420, 360 * rows),
        margin=dict(t=70, b=40, l=40, r=40),
        legend=dict(orientation='h', y=-0.05),
    )

    with model_plot_out:
        clear_output(wait=True)
        display(fig)

    state['model_plot_key'] = key
    state['last_plot_run_dir'] = str(run_dir)


def stop_auto_refresh():
    state['refresh_loop'] = False
    t = state.get('monitor_thread')
    if t is not None and t.is_alive():
        t.join(timeout=0.2)
    state['monitor_thread'] = None


def get_progress_event_signature(run_dir):
    if run_dir is None:
        return None
    p = Path(run_dir) / 'progress.log'
    if not p.exists():
        return None

    lines = p.read_text(errors='replace').splitlines()
    events = [ln.strip() for ln in lines if ln.startswith('Linesearch') or ln.startswith('Iteration')]
    if not events:
        return ('none', 0)
    return (events[-1], len(events))


def refresh_run_status(_=None, refresh_images=True):
    if state.get('refresh_in_progress'):
        return

    state['refresh_in_progress'] = True
    try:
        proc = state.get('process')
        run_dir = state.get('active_run_dir')
        if proc is None:
            if run_dir is None:
                run_status.value = 'No active inversion process in this session.'
            else:
                run_status.value = f'No active process. Last run folder: {run_dir.name}'
        else:
            rc = proc.poll()
            if rc is None:
                run_status.value = f'Inversion is running (pid={proc.pid}) in {run_dir.name}.'
            else:
                run_status.value = f'Inversion finished with exit code {rc} in {run_dir.name}.'
                state['process'] = None
                state['refresh_loop'] = False

        log_out.value = read_tail(state.get('active_run_log'), n_lines=60)
        progress_out.value = parse_progress_summary(run_dir)
        job_progress_out.value, mpiqueue_out.value = parse_mpiqueue_summary(run_dir)
        worker_logs_out.value = parse_worker_logs_summary(run_dir)
        ls_model_out.value = summarize_linesearch_model(run_dir)

        current_event = get_progress_event_signature(run_dir)
        if current_event != state.get('last_progress_event'):
            state['last_progress_event'] = current_event
            if refresh_images and not state.get('initial_load'):
                update_model_plot(run_dir, force=True)
            if current_event is not None and run_dir is not None:
                push_message(f'Progress event detected in {Path(run_dir).name}: {current_event[0]}')
    finally:
        state['refresh_in_progress'] = False


def start_auto_refresh(interval_s=5.0):
    stop_auto_refresh()
    state['refresh_loop'] = True

    def _loop():
        while state.get('refresh_loop'):
            try:
                refresh_run_status(refresh_images=False)
            except Exception:
                pass
            time.sleep(interval_s)

    t = threading.Thread(target=_loop, daemon=True)
    t.start()
    state['monitor_thread'] = t


def initialize_active_run():
    latest = find_latest_run_dir(INV_2D_RUNS_DIR)
    if latest is None:
        return
    state['ladder_run_dir'] = latest
    stages = iter_stages(latest)
    active = stages[-1]['path'] if stages else None
    if active is None:
        return
    state['active_run_dir'] = active
    preferred_logs = [active / 'inversion.log', active / 'inversion_run.log']
    for lp in preferred_logs:
        if lp.exists():
            state['active_run_log'] = lp
            break
    if state['active_run_log'] is None:
        state['active_run_log'] = preferred_logs[0]



In [ ]:
title = ipw.HTML('<h2>03_2d_inversion</h2>')

initial_model_mode = ipw.Dropdown(
    options=[
        ('Uniform conductivity model', 'uniform_conductivity'),
        ('Uniform resistivity model (converted to conductivity)', 'uniform_resistivity'),
    ],
    value='uniform_resistivity',
    description='Initial model:',
    layout=ipw.Layout(width='700px'),
)

uniform_cond = ipw.FloatText(value=0.01, description='sigma (S/m):', layout=ipw.Layout(width='320px'))
uniform_rho = ipw.FloatText(value=100.0, description='rho (Ohm.m):', layout=ipw.Layout(width='320px'))
constrain_bounds = ipw.Checkbox(value=True, description='Constrain sg bounds')
sg_min_value = ipw.FloatText(value=1e-8, description='sg min (S/m):', layout=ipw.Layout(width='320px'))
sg_max_value = ipw.FloatText(value=1.0, description='sg max (S/m):', layout=ipw.Layout(width='320px'))
max_iterations = ipw.IntText(value=20, description='Max iter:', layout=ipw.Layout(width='220px'))
geps_value = ipw.FloatText(value=0.0, description='geps (0=off):', layout=ipw.Layout(width='220px'))
# Default matches the forward run's own apertx (from 01_fw_setup's design_
# explicit_fd) when available, so the FWI's local model isn't accidentally
# narrower than what the forward run used - overridable if needed.

# NO DATASET SELECTOR (RULE 2): a dataset is named `f1000Hz_hx`, so choosing one
# is choosing a frequency and a source. Every action here stages and runs ALL of
# them, so there is nothing to select. The sources of a frequency are not even
# separate runs any more - they are one joint inversion - so the only sequence
# left is over frequencies, lowest first, each later one starting from the
# previous inverted model. Step 01 already fixed which frequencies exist. `DS`
# (bound in cell 1) points at the dataset whose grid the batch is currently
# staging.
_fwd_meta_for_defaults = json.loads(DS.setup_meta.read_text()) if DS.setup_meta.exists() else {}
apertx_value = ipw.FloatText(value=float(_fwd_meta_for_defaults.get('apertx_m', 60.0)), description='apertx (m):', layout=ipw.Layout(width='220px'))
dtx_value = ipw.FloatText(value=6.0, description='dtx:', layout=ipw.Layout(width='220px'))
dtz_value = ipw.FloatText(value=6.0, description='dtz:', layout=ipw.Layout(width='220px'))
tik_sgregalpha_value = ipw.FloatText(value=0.0, description='Tikhonov (sg):', step=0.001, layout=ipw.Layout(width='220px'))
clean_run_dir = ipw.Checkbox(value=True, description='Clean scale directory before staging')
run_selector = ipw.Dropdown(options=[('No runs found', None)], value=None, description='Load run:', layout=ipw.Layout(width='420px'))

nproc_inv_input = ipw.BoundedIntText(
    value=min(CONFIG.nproc_default, MAX_CPUS), min=2, max=MAX_CPUS,
    description='nproc', layout=ipw.Layout(width='200px'),
)
gen_inputs_btn = ipw.Button(description='Generate inversion inputs', button_style='primary')
run_inv_btn = ipw.Button(description='Run inversion locally', button_style='success')
refresh_btn = ipw.Button(description='Refresh from progress.log')
stop_btn = ipw.Button(description='Stop run', button_style='warning')
quit_btn = ipw.Button(description='Quit GUI server', button_style='danger')

run_status = ipw.HTML(value='No active inversion process in this session.')
status_out = ipw.Textarea(value='', description='Status:', layout=ipw.Layout(width='100%', height='200px'))
log_out = ipw.Textarea(value='No log file yet.', description='Run log:', layout=ipw.Layout(width='100%', height='220px'))
progress_out = ipw.Textarea(value='No progress information yet.', description='Progress:', layout=ipw.Layout(width='100%', height='160px'))
job_progress_out = ipw.Textarea(value='No queue information yet.', description='Queue summary:', layout=ipw.Layout(width='100%', height='120px'))
mpiqueue_out = ipw.Textarea(value='No mpiqueue tail yet.', description='mpiqueue tail:', layout=ipw.Layout(width='100%', height='220px'))
worker_logs_out = ipw.Textarea(value='No worker log information yet.', description='Worker logs:', layout=ipw.Layout(width='100%', height='240px'))
ls_model_out = ipw.Textarea(value='No linesearch model yet.', description='LS model:', layout=ipw.Layout(width='100%', height='140px'))
model_plot_out = ipw.Output(layout=ipw.Layout(width='100%', border='1px solid #ddd', height='1200px', overflow='auto'))


def live_inversion_controls():
    """The inv.cfg knobs the engine reads, taken from the live widgets.

    These must be written at RUN time as well as at Generate Inputs.
    Staging copies `input/<freq>/inv.cfg` into the scale subdirectory;
    if that copy is left unchanged, Max iter / geps / dtx / Tikhonov are
    whatever was typed when the inputs were generated, not the widgets.
    """
    return {
        'max_iterations': int(max_iterations.value),
        'apertx': float(apertx_value.value),
        'dtx': float(dtx_value.value),
        'dtz': float(dtz_value.value),
        'tik_sgregalpha': float(tik_sgregalpha_value.value),
        'geps': float(geps_value.value),
    }


def _selected_model_kwargs():
    mode = initial_model_mode.value
    if mode == 'uniform_conductivity':
        return {
            'initial_model_mode': mode,
            'uniform_conductivity': float(uniform_cond.value),
            'uniform_resistivity': None,
        }
    return {
        'initial_model_mode': mode,
        'uniform_conductivity': None,
        'uniform_resistivity': float(uniform_rho.value),
    }


def read_forward_setup_metadata():
    path = DS.setup_meta
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text())
    except Exception:
        return {}


def write_setup_metadata(created_files, forward_meta=None, output_dir=None, forward_dirs=None):
    forward_meta = dict(forward_meta or {})
    forward_data_dim = int(forward_meta.get('forward_data_dim', 2))
    out_dir = Path(output_dir) if output_dir else INV_2D_INPUT_DIR
    forward_dirs = dict(forward_dirs or {})
    metadata = {
        'input_dir': str(out_dir),
        'run_dir': str(state.get('active_run_dir')) if state.get('active_run_dir') else None,
        'fdmodel_dir': str(DS.dir),
        # Every forward run this inversion fits, one per source component. A
        # joint run has more than one, and `fdmodel_dir` above is only the first.
        'fdmodel_dirs': {k: str(v) for k, v in forward_dirs.items()},
        'source_fields': sorted(forward_dirs),
        'initial_model_mode': initial_model_mode.value,
        'max_iterations': int(max_iterations.value),
        'apertx': float(apertx_value.value),
        'dtx': float(dtx_value.value),
        'dtz': float(dtz_value.value),
        'tik_sgregalpha': float(tik_sgregalpha_value.value),
        'geps': float(geps_value.value),
        'constrain': bool(constrain_bounds.value),
        'sg_min': float(sg_min_value.value),
        'sg_max': float(sg_max_value.value),
        'files': {k: str(v) for k, v in created_files.items()},
        'forward_data_dim': forward_data_dim,
        'forward_engine': str(forward_meta.get('forward_engine', CONFIG.forward_engine_te2d())),
        'forward_cfg': str(forward_meta.get('forward_cfg', 'mod.cfg')),
        'dimensional_warning_issued': bool(forward_data_dim == 3),
    }
    (out_dir / 'inversion_setup_metadata.json').write_text(json.dumps(metadata, indent=2))


def set_active_run(run_dir):
    if run_dir is None:
        state['active_run_dir'] = None
        state['active_run_log'] = None
        state['last_progress_event'] = None
        return
    run_dir = Path(run_dir)
    state['active_run_dir'] = run_dir
    preferred_logs = [run_dir / 'inversion.log', run_dir / 'inversion_run.log']
    chosen = None
    for lp in preferred_logs:
        if lp.exists():
            chosen = lp
            break
    state['active_run_log'] = chosen if chosen is not None else preferred_logs[0]
    state['model_plot_key'] = None
    state['last_plot_run_dir'] = None
    state['last_progress_event'] = None


def refresh_run_selector(select_latest=False):
    runs = list_run_dirs(INV_2D_RUNS_DIR)
    options = [(ladder_run_label(p), str(p)) for _, p in runs]
    if not options:
        options = [('No runs found', None)]

    current = None
    if state.get('ladder_run_dir') is not None:
        current = str(state['ladder_run_dir'])

    state['updating_run_selector'] = True
    run_selector.options = options
    if select_latest and runs:
        run_selector.value = str(runs[-1][1])
    elif current in [v for _, v in options]:
        run_selector.value = current
    else:
        run_selector.value = options[0][1]
    state['updating_run_selector'] = False


def on_select_run(change):
    if change.get('name') != 'value' or state.get('updating_run_selector'):
        return
    selected = change.get('new')
    if not selected:
        return

    proc = state.get('process')
    if proc is not None and proc.poll() is None:
        push_message('Cannot switch run while inversion is active. Stop it first.')
        state['updating_run_selector'] = True
        run_selector.value = str(state['ladder_run_dir']) if state.get('ladder_run_dir') else None
        state['updating_run_selector'] = False
        return

    ladder = Path(selected)
    state['ladder_run_dir'] = ladder
    stages = iter_stages(ladder)
    set_active_run(stages[-1]['path'] if stages else None)
    update_info_panel()
    refresh_run_status()
    push_message(f'Loaded ladder {ladder.name}'
                 + (f' (scale {stages[-1]["name"]})' if stages else ''))


def dataset_input_dir(name):
    """Inversion input directory for one unit of the matrix."""
    return INV_2D_INPUT_DIR if name is None else INV_2D_INPUT_DIR / str(name)


def on_generate_inputs(_):
    """Stage inversion inputs for EVERY frequency of the matrix.

    One input directory per frequency, holding all of that frequency's sources:
    `f2000Hz_hx_hz/` with four record files and `source_type = "3,5"`. Which
    frequencies get staged is not a choice the operator makes here - Step 01
    decided that.

    Every directory gets a uniform `sg0.rss`. That is the start of the FIRST
    frequency. Later frequencies receive the previous inverted model at RUN
    time, via `handoff_starting_model`, because that model does not exist yet
    when the inputs are staged.
    """
    try:
        if not INV_TEMPLATE.exists():
            raise FileNotFoundError(f'Missing inversion template: {INV_TEMPLATE}')
        groups = available_groups()
        if not groups:
            raise FileNotFoundError(f'No forward datasets under {FWD_ROOT}. Run Step 01 first.')
        made = []
        for i, g in enumerate(groups, 1):
            _select_dataset(g['datasets'][0]['name'])
            push_message(f"[{i}/{len(groups)}] staging joint inversion inputs for "
                         f"{g['name']} ({'+'.join(g['sources'])})")
            _generate_inputs_for_active(dataset_input_dir(g['name']), g['forward_dirs'])
            made.append(g['name'])
        _select_dataset(None)
        push_message(f'Staged inversion inputs for {len(made)} frequency/frequencies: '
                     + ', '.join(made))
    except Exception as exc:
        push_message(f'Generate inversion inputs failed: {exc}')
        push_message(traceback.format_exc())


def _generate_inputs_for_active(out_dir, forward_dirs=None):
    try:
        if not DS.dir.exists():
            raise FileNotFoundError(f'Missing forward dataset folder: {DS.dir}')

        forward_meta = read_forward_setup_metadata()
        forward_data_dim = int(forward_meta.get('forward_data_dim', 2))

        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        # A dict of {source_field: forward_dir} makes the run JOINT over those
        # sources; a bare path is the single-source case of the same call.
        created = prepare_inversion_inputs(
            fdmodel_dir=forward_dirs or DS.dir,
            template_cfg=INV_TEMPLATE,
            output_dir=out_dir,
            constrain=bool(constrain_bounds.value),
            sg_min=float(sg_min_value.value),
            sg_max=float(sg_max_value.value),
            **live_inversion_controls(),
            **_selected_model_kwargs(),
        )

        if forward_data_dim == 3:
            push_message(
                'WARNING: Input data was modeled in 3D, but this inversion engine is 2D. '
                'Results are expected to be poor.'
            )

        write_setup_metadata(created, forward_meta=forward_meta, output_dir=out_dir,
                             forward_dirs=forward_dirs)
        push_message(f'  inputs -> {out_dir}')
        update_info_panel()
    except Exception:
        # Re-raised so the batch loop reports WHICH dataset failed instead of
        # logging a detached traceback and carrying on as if it had worked.
        raise


def on_run_inversion(_):
    """Run ONE frequency ladder: every tone, jointly over its sources.

    Allocates a single `Run{N}/`. Each frequency is inverted in
    `Run{N}/<group>/`, which is the engine cwd. One `inv.cfg` per scale with
    `source_type = "3,5"` and four `Recordfile_<SRC>_<REC>` keys, so the Kx
    and Kz gradients are summed before the model is stepped.

    Sequential because each scale already uses every core through mpirun, AND
    because each frequency after the first starts from the previous inverted
    model: `handoff_starting_model` resamples `Results/sg_up.rss-<iter>` onto
    this frequency's grid as `sg0.rss`. Staging writes a uniform `sg0` into
    every input directory; that is the first-frequency start.
    """
    try:
        if state.get('process') is not None and state['process'].poll() is None:
            raise RuntimeError('Inversion process already running. Stop it first.')
        if not RUNINV_TEMPLATE.exists():
            raise FileNotFoundError(f'Missing run script template: {RUNINV_TEMPLATE}')
        ds = available_groups()
        if not ds:
            raise FileNotFoundError(f'No forward datasets under {FWD_ROOT}. Run Step 01 first.')
        missing = [d['name'] for d in ds
                   if not (dataset_input_dir(d['name']) / 'inv.cfg').exists()]
        if missing:
            raise FileNotFoundError(
                'Missing inv.cfg for: ' + ', '.join(missing)
                + '. Press "Generate inversion inputs" first.')

        nproc = max(2, min(int(getattr(nproc_inv_input, 'value', 4)), MAX_CPUS))
        state['batch_stop'] = False

        def _worker():
            ladder_dir = allocate_ladder_run(INV_2D_RUNS_DIR)
            write_ladder_manifest(
                ladder_dir, empty_ladder_manifest(source_fields=ds[0]['sources']))
            state['ladder_run_dir'] = ladder_dir
            prev_model = None
            for i_ds, d in enumerate(ds, 1):
                if state.get('batch_stop'):
                    push_message('Ladder stopped before ' + d['name'])
                    break
                try:
                    run_dir = stage_cwd(ladder_dir, d['name'])
                    run_dir.mkdir(parents=True, exist_ok=True)
                    staged = stage_run_directory(
                        input_dir=dataset_input_dir(d['name']),
                        run_dir=run_dir,
                        clean=bool(clean_run_dir.value),
                        include_patterns=['*'],
                    )
                    template_sg = Path(next(iter(d['forward_dirs'].values()))) / 'sg.rss'
                    report = handoff_starting_model(
                        prev_model, template_sg, run_dir / 'sg0.rss')
                    if report is None:
                        (run_dir / 'sg0_from.txt').write_text('uniform\n')
                        start_msg = 'starting model: uniform'
                        sg0_from = 'uniform'
                    else:
                        (run_dir / 'sg0_from.txt').write_text(str(prev_model) + '\n')
                        start_msg = (
                            f'starting model: previous inverted model resampled onto '
                            f'{d["name"]} grid (dx {report["dst_dx"]:.2f} m)'
                        )
                        sg0_from = Path(prev_model).parent.name
                    # Live widgets, not the values baked when inputs were generated.
                    ctrl = live_inversion_controls()
                    apply_inversion_controls(run_dir / 'inv.cfg', **ctrl)
                    stop_msg = (
                        f'max_iter={ctrl["max_iterations"]}'
                        + (f', geps={ctrl["geps"]:g}' if ctrl['geps'] > 0 else ', geps off')
                    )
                    run_script = run_dir / 'runinv.sh'
                    run_script.write_text(
                        patch_runinv_template(RUNINV_TEMPLATE.read_text(), CONFIG, nproc))
                    (run_dir / 'dataset.txt').write_text(str(d['name']) + '\n')
                    upsert_stage(ladder_dir, {
                        'index': i_ds - 1,
                        'freq_hz': float(d['freq_hz']),
                        'name': d['name'],
                        'dir': d['name'],
                        'sg0_from': sg0_from,
                        'status': 'running',
                        'source_fields': d['sources'],
                    })

                    run_log = run_dir / 'inversion.log'
                    with open(run_log, 'w') as logf:
                        proc = subprocess.Popen(
                            ['sh', str(run_script)], cwd=str(run_dir),
                            stdout=logf, stderr=subprocess.STDOUT,
                        )
                    set_active_run(run_dir)
                    refresh_run_selector()
                    state['active_run_log'] = run_log
                    state['process'] = proc
                    state['last_sg_ls_mtime'] = None
                    state['last_progress_event'] = ('__new_run__', -1)
                    push_message(
                        f'[{i_ds}/{len(ds)}] {d["name"]}: staged {len(staged)} file(s) into '
                        f'{ladder_dir.name}/{run_dir.name}/, pid={proc.pid}; '
                        f'{start_msg}; {stop_msg}')
                    rc = proc.wait()
                    stopped = bool(state.get('batch_stop'))
                    status = 'stopped' if stopped else ('ok' if rc == 0 else 'failed')
                    upsert_stage(ladder_dir, {
                        'index': i_ds - 1,
                        'freq_hz': float(d['freq_hz']),
                        'name': d['name'],
                        'dir': d['name'],
                        'sg0_from': sg0_from,
                        'status': status,
                        'source_fields': d['sources'],
                    })
                    push_message(f'[{i_ds}/{len(ds)}] {d["name"]}: finished rc={rc} '
                                 f'({ladder_dir.name}/{run_dir.name})')
                    if stopped or rc != 0:
                        push_message(
                            'Ladder stopped: later frequencies need this inverted '
                            'model as their starting model.')
                        break
                    mdl = find_output_model(run_dir)
                    if mdl is None:
                        upsert_stage(ladder_dir, {
                            'index': i_ds - 1,
                            'freq_hz': float(d['freq_hz']),
                            'name': d['name'],
                            'dir': d['name'],
                            'sg0_from': sg0_from,
                            'status': 'failed',
                            'source_fields': d['sources'],
                        })
                        push_message(
                            'Ladder stopped: no inverted model in Results/ to hand '
                            f'to the next frequency ({ladder_dir.name}/{run_dir.name}).')
                        break
                    prev_model = mdl
                except Exception as exc:
                    push_message(f'[{i_ds}/{len(ds)}] {d["name"]}: FAILED {exc}')
                    push_message(
                        'Ladder stopped: later frequencies need this inverted '
                        'model as their starting model.')
                    break
            state['process'] = None
            push_message('Frequency ladder finished.')

        threading.Thread(target=_worker, daemon=True).start()
        push_message(f'Started sequential inversion of {len(ds)} frequency/frequencies '
                     '(each joint over its source components; each later frequency '
                     'starts from the previous inverted model).')
        start_auto_refresh()
    except Exception as exc:
        push_message(f'Run inversion failed: {exc}')
        push_message(traceback.format_exc())

def on_refresh(_):
    # Pick up run folders created since the GUI loaded - including the ones
    # this session just produced - before reporting on the selected one.
    refresh_run_selector()
    refresh_run_status()
    # Ensure manual refresh always re-evaluates model panel for selected run.
    update_model_plot(state.get('active_run_dir'), force=True)


def on_stop(_):
    # Stop the whole sequential batch, not just the run currently executing -
    # otherwise pressing Stop starts the next dataset.
    state['batch_stop'] = True
    try:
        proc = state.get('process')
        if proc is None or proc.poll() is not None:
            push_message('No active inversion process to stop.')
            refresh_run_status()
            return
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
        push_message('Stopped inversion process.')
    except Exception as exc:
        push_message(f'ERROR stopping inversion: {exc}')
    finally:
        refresh_run_status()


def on_quit_server(_):
    try:
        stop_auto_refresh()
        proc = state.get('process')
        if proc is not None and proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
        pid = None
        if VOILA_PID_FILE.exists():
            pid = int(VOILA_PID_FILE.read_text().strip())
        push_message('Shutting down inversion GUI server...')
        if pid:
            os.kill(pid, signal.SIGTERM)
    except Exception as exc:
        push_message(f'ERROR quitting GUI server: {exc}')


bind_button_with_feedback(gen_inputs_btn, on_generate_inputs, 'Generating inversion inputs')
bind_button_with_feedback(run_inv_btn, on_run_inversion, 'Starting inversion run')
bind_button_with_feedback(refresh_btn, on_refresh, 'Refreshing inversion status')
bind_button_with_feedback(stop_btn, on_stop, 'Stopping inversion run')
bind_button_with_feedback(quit_btn, on_quit_server, 'Shutting down inversion GUI server')
run_selector.observe(on_select_run, names='value')

controls1 = ipw.HBox([max_iterations, geps_value, apertx_value, dtx_value, dtz_value, tik_sgregalpha_value])
controls2 = ipw.HBox([uniform_cond, uniform_rho])
controls3 = ipw.HBox([constrain_bounds, sg_min_value, sg_max_value])
run_controls = ipw.HBox([run_selector])
buttons = ipw.HBox([nproc_inv_input, gen_inputs_btn, run_inv_btn, stop_btn, refresh_btn, quit_btn])

info = ipw.HTML(value='')
update_info_panel()

layout = ipw.VBox([
    title,
    info,
    initial_model_mode,
    controls1,
    controls2,
    controls3,
    run_controls,
    clean_run_dir,
    buttons,
    run_status,
    progress_out,
    job_progress_out,
    ls_model_out,
    model_plot_out,
    worker_logs_out,
    mpiqueue_out,
    status_out,
    log_out,
])

state['initial_load'] = True
initialize_active_run()
refresh_run_selector(select_latest=True)
update_info_panel()
display(layout)
refresh_run_status(refresh_images=False)
state['initial_load'] = False

# Auto-refresh is NOT started on load (avoids Voila hang). It starts only when you
# click "Run inversion locally". The background thread only updates log text, never
# the model plot; use "Refresh from progress.log" to update the plot.
